In [2]:
!pip install -q --no-cache-dir "vllm==0.19.1"
!pip -q install -U transformers huggingface_hub
!pip install -U "protobuf>=5.26.1,<6"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 273.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 381.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 267.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 327.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 349.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 339.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 280.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 359.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 366.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 309.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 350.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import argparse
import os
import sys
from pathlib import Path
import time

ROOT = "/content/drive/MyDrive/Colab Notebooks/kisti/system/company_to_project"
os.chdir(ROOT)
sys.path.insert(0, ROOT)


def next_result_path(base_path="./result/result.csv"):
    base = Path(base_path)
    base.parent.mkdir(parents=True, exist_ok=True)

    if not base.exists():
        return str(base)

    stem = base.stem
    suffix = base.suffix
    parent = base.parent

    idx = 2
    while True:
        candidate = parent / f"{stem}_{idx}{suffix}"
        if not candidate.exists():
            return str(candidate)
        idx += 1

def format_time(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = seconds % 60

    if hours > 0:
        return f"{hours}시간 {minutes}분 {secs:.2f}초"

    if minutes > 0:
        return f"{minutes}분 {secs:.2f}초"

    return f"{secs:.2f}초"


def main():
    print("모듈 및 모델 라이브러리 import 중 . . .")
    from data_pipeline import (
        load_data,
        select_company_matches,
        preprocess,
        load_embedding_model,
    )
    from llm_pipeline import load_model, run_generation
    print("import 완료")

    print("data.csv 로드 중 . . .")
    df = load_data("./data/data.csv")
    print("data.csv 로드 완료")

    print("임베딩 모델 로딩 중 . . .")
    t_embed_model_start = time.perf_counter()
    embed_model, embed_tokenizer, embed_device = load_embedding_model("BAAI/bge-m3")
    t_embed_model_end = time.perf_counter()
    print(f"[완료] 임베딩 모델 로딩: {format_time(t_embed_model_end - t_embed_model_start)}")

    print("LLM 모델 로딩 중 . . .")
    t_llm_model_start = time.perf_counter()
    llm, llm_tokenizer = load_model(
        model_id="Qwen/Qwen3.5-35B-A3B-GPTQ-Int4", #"./models/Qwen3.5-35B-A3B-GPTQ-Int4",
        hf_token=os.environ.get("HF_TOKEN"),
        tensor_parallel_size=1,
    )
    t_llm_model_end = time.perf_counter()
    print(f"[완료] LLM 모델 로딩: {format_time(t_llm_model_end - t_llm_model_start)}")

    while True:
        company_name = input("회사명을 입력하세요. 종료하려면 end 입력: ").strip()

        if company_name.lower() == "end":
            print("종료합니다.")
            break

        try:
            top_n = int(input("가져올 개수를 입력하세요: ").strip())
        except:
            print("숫자를 입력해주세요.")
            continue

        try:
          print("데이터 전처리 시작")
          t_pre_start = time.perf_counter()

          tmp = select_company_matches(
              df=df,
              company_name=company_name,
              top_n=top_n,
          )

          tmp = preprocess(
              tmp,
              embed_model=embed_model,
              embed_tokenizer=embed_tokenizer,
              embed_device=embed_device,
          )

          tmp_path = next_result_path("./data/tmp.csv")
          tmp.to_csv(tmp_path, index=False, encoding="utf-8-sig")

          t_pre_end = time.perf_counter()
          elapsed = t_pre_end - t_pre_start
          print(f"[완료] 데이터 전처리 완료: {format_time(elapsed)}")

          print(f"데이터 전처리 수: {len(tmp)}건")
          print(f"전처리 파일 저장: {tmp_path}")

          out_path = next_result_path("./result/result.csv")

          print(f"LLM 생성 시작: {out_path}")
          t_llm_start = time.perf_counter()

          result = run_generation(
              tmp=tmp,
              model=llm,
              tokenizer=llm_tokenizer,
              out_path=out_path,
              overwrite=False,
          )

          t_llm_end = time.perf_counter()
          elapsed = t_llm_end - t_llm_start
          print(f"[완료] llm 생성 완료: {format_time(elapsed)}")
          print(f"결과 파일: {result['out_path']}")

        except ValueError as e:
            print(f"[오류] {e}")
            print("다른 회사명을 입력하거나 end로 종료해주세요.")
            continue

        except Exception as e:
            print(f"[오류] 처리 중 문제가 발생했습니다: {e}")
            print("다른 회사명을 입력하거나 end로 종료해주세요.")
            continue


main()

모듈 및 모델 라이브러리 import 중 . . .
import 완료
data.csv 로드 중 . . .
data.csv 로드 완료
임베딩 모델 로딩 중 . . .


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

[완료] llm 생성 완료: 15.34초
결과 파일: result/result.csv
회사명을 입력하세요. 종료하려면 end 입력: 나노시스템
가져올 개수를 입력하세요: 2
데이터 전처리 시작
[완료] 데이터 전처리 완료: 0.17초
데이터 전처리 수: 2건
전처리 파일 저장: data/tmp_2.csv
LLM 생성 시작: result/result_2.csv


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]